# Get started with `shrecc.NewDatabase`

**This is the shortest complete workflow for writing a historical electricity database. Before running it, create a Brightway project containing a basic ecoinvent (for these purposes, we use ecoinvent 3.11) background database.**

In [ ]:
from shrecc import NewDatabase
import bw2data as bd
import bw2calc as bc
import pandas as pd

## Configure

Edit the project, background database, output name, countries, and year below. With no time selection, SHRECC uses the complete year. A historical year automatically selects Energy Charts as the source.

We will use a simple case of electric vehicle charging during 4 different time slots in a year: during the day and night in winter and during the day and night in summer.

In [ ]:
# My parameters

years        = 2026 # Either an int or a list
countries = [
    "CZ",
    "DK",
    "FR",
    "GR"
]
project_name = "SHRECCei311" # The name of my Brightway project, where to write the electricity database
bg_db_name   = "ecoinvent-3.11-cutoff" # The name of the background database with which electricity inventories are linked
my_db_name   = "shrecc_2026_winter_day" # The name of my new electricity database
time_range   = ["2026-01-01 00:00", "2026-03-31 23:00"] # Interval in which you want to create the historical database
hour_range   = [9,17] # Hour range which you want to select

# My NewDatabase object

winter_day = NewDatabase(
    years=years,
    countries=countries,
    project_name=project_name,
    bg_db_name=bg_db_name,
    my_db_name=my_db_name,
    time_range=time_range,
    hour_range=hour_range,
    verbose=True,
    download=True
)

## Create

`create()` obtains or loads the source data, solves the interconnected electricity system, maps activities, and prepares the database table. It does not modify Brightway.

In [ ]:
winter_day.create()

## Write

`write()` writes the prepared inventory to the configured Brightway project. An existing output database with the same name is replaced.

In [ ]:
winter_day.write()

In [ ]:
winter_day.written_database_names

In [ ]:
bd.databases

## Lets automate this

In [ ]:
years        = 2026
countries = ["CZ", "DK", "FR", "GR"]
project_name = "SHRECCei311"
bg_db_name   = "ecoinvent-3.11-cutoff"

configs = [
    {
        "my_db_name": "shrecc_2026_winter_night",
        "time_range": ["2026-01-01 00:00", "2026-03-31 23:00"],
        "hour_range": [0, 8],
    },
    {
        "my_db_name": "shrecc_2026_summer_day",
        "time_range": ["2026-06-01 00:00", "2026-08-31 23:00"],
        "hour_range": [9, 17],
    },
    {
        "my_db_name": "shrecc_2026_summer_night",
        "time_range": ["2026-06-01 00:00", "2026-08-31 23:00"],
        "hour_range": [0, 8],
    },
]

databases = {}
written_names = {}

for cfg in configs:
    print(f"Building {cfg['my_db_name']} ...")
    db = NewDatabase(
        years=years,
        countries=countries,
        project_name=project_name,
        bg_db_name=bg_db_name,
        my_db_name=cfg["my_db_name"],
        time_range=cfg["time_range"],
        hour_range=cfg["hour_range"],
        verbose=True,
        download=True,
    )
    db.create()
    db.write()
    databases[cfg["my_db_name"]] = db
    written_names[cfg["my_db_name"]] = db.written_database_names
    print(f"  -> written as: {db.written_database_names}")

print(written_names)

## Impact assessment

Let's assume a slow charging rate of 3.7 kW for 8 h, using 29.6 kWh of low voltage electricity

In [ ]:
ef_methods = [m for m in bd.methods if m[1] == "EF v3.0"]
config = {"impact_categories": ef_methods}

In [ ]:
demands_summer_day = {
    f"{a['name']}, {a['location']}, summer, day": {a["id"]: 29.6}
    for a in bd.Database("shrecc_2026_summer_day")
}
demands_summer_night = {
    f"{a['name']}, {a['location']}, summer, night": {a["id"]: 29.6}
    for a in bd.Database("shrecc_2026_summer_night")
}
demands_winter_day = {
    f"{a['name']}, {a['location']}, winter, day": {a["id"]: 29.6}
    for a in bd.Database("shrecc_2026_winter_day")
}
demands_winter_night = {
    f"{a['name']}, {a['location']}, winter, night": {a["id"]: 29.6}
    for a in bd.Database("shrecc_2026_winter_night")
}

In [ ]:
demands = {**demands_summer_day, **demands_summer_night, **demands_winter_day, **demands_winter_night}
demands

In [ ]:
acts_ei = [
    bd.get_node(
        database="ecoinvent-3.11-cutoff",
        location=country,
        name="market for electricity, low voltage",
    )
    for country in sorted(countries)
]
acts_ei

In [ ]:
demands.update({f"{a['name']}, {a['location']}": {a["id"]: 29.6} for a in acts_ei})
demands

In [ ]:
data_objs = bd.get_multilca_data_objs(functional_units=demands, method_config=config)

In [ ]:
m_lca = bc.MultiLCA(demands=demands, method_config=config, data_objs=data_objs)
m_lca.lci()
m_lca.lcia()

In [ ]:
results = []
for (method, fu), score in m_lca.scores.items():
    demand = m_lca.demands[fu]
    a_id = list(demand).pop()
    a = bd.get_activity(a_id)
    results.append(
        {
            "activity": a["name"],
            "database": a["database"],
            "location": a["location"],
            "production amount": demand[a_id],
            "activity unit": a["unit"],
            "reference product": a["reference product"],
            "methodology": method[0],
            "category": method[1],
            "indicator": method[2],
            "score": score,
            "unit": bd.Method(method).metadata["unit"],
        },
    )
res_df = pd.DataFrame(results)
res_df = res_df.set_index([c for c in res_df.columns if c != "score"]).unstack(
    ["category", "indicator", "unit"]
)

In [ ]:
import matplotlib.pyplot as plt

indicators = [
    (("score", "EF v3.0", "climate change", "kg CO2-Eq"), "kg CO2 eq./kWh"),
    (("score", "EF v3.0", "material resources: metals/minerals", "kg Sb-Eq"), "kg Sb eq./kWh"),
    (("score", "EF v3.0", "land use", "dimensionless"), "dimensionless / kWh"),
]

res_df.sort_index(level="location", inplace=True)

color_map = {
    "ecoinvent-3.11-cutoff": "#2B2B2B",
    "shrecc_2026_summer_day": "#F5C08A",
    "shrecc_2026_summer_night": "#E07A2F",
    "shrecc_2026_winter_day": "#A8D4E3",
    "shrecc_2026_winter_night": "#2C6E9E",  
}

fig, axes = plt.subplots(len(indicators), 1, figsize=(10, 15), sharex=True)

for ax, (col, ylabel) in zip(axes, indicators):
    data = res_df[col].droplevel(["activity"]).unstack(["location"])
    databases_in_order = data.index.get_level_values("database").unique()
    ordered_colors = [color_map[db] for db in databases_in_order]
    data.T.plot.bar(ax=ax, legend=False, color=ordered_colors)
    ax.set_ylabel(ylabel)
    ax.set_title(col[2])

axes[-1].tick_params(axis="x", rotation=45)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    ncol=1,
    fontsize=9,
    frameon=False,
)

plt.tight_layout()
plt.subplots_adjust(right=0.78)
plt.show()

## Full year

In [ ]:
years        = 2026 # Either an int or a list
countries = [
    "CZ",
    "DK",
    "FR",
    "GR"
]
project_name = "SHRECCei311" # The name of my Brightway project, where to write the electricity database
bg_db_name   = "ecoinvent-3.11-cutoff" # The name of the background database with which electricity inventories are linked
my_db_name   = "shrecc_2026" # The name of my new electricity database
time_range   = ["2026-01-01 00:00", "2026-08-31 23:00"]

# My NewDatabase object

electricity_database = NewDatabase(
    years=years,
    countries=countries,
    project_name=project_name,
    bg_db_name=bg_db_name,
    my_db_name=my_db_name,
    time_range=time_range,
    verbose=True,
    download=True
)

In [ ]:
electricity_database.create()

In [ ]:
electricity_database.write()

In [ ]:
electricity_database.written_database_names

In [ ]:
demands = {act["location"]:{act["id"]:1} for act in bd.Database("shrecc_2026")}

In [ ]:
method = {"impact_categories":[('ecoinvent-3.11',
  'EF v3.1',
  'climate change',
  'global warming potential (GWP100)')]}

In [ ]:
electricity_database.lcia(methods=method["impact_categories"])

In [ ]:
electricity_database.lcia_results.hourly().to_dataframe()

In [ ]:
electricity_database.lcia_results.hourly().to_dataframe().droplevel("impact_category")["intensity"].unstack("consumer_country").plot(figsize=(16,6))

In [ ]:
cons_mix = electricity_database.results(2026)["consumption_mix_volume"].sel(consumer_country=countries).to_dataframe().unstack("consumer_country")
cons_mix.groupby("technology").sum()

In [ ]:
top10 = cons_mix.groupby("source_country").sum()[("consumption_mix_volume", 'DK')].nlargest(10).index

In [ ]:
cons_mix.groupby(["time", "technology"]).sum()[("consumption_mix_volume", "DK")].unstack().plot.area(figsize=(16,6), linewidth=0, colormap="tab20")

In [ ]:
electricity_database.mapping_gap_reports

In [ ]:
electricity_database.ecoinvent_mapping